In [98]:
import re
import unicodedata

FILE_TO_PROCESS = "nova-zemya.txt"

In [99]:
def clean_from_chitanka(text: str) -> str:
    # Strip start
    match = re.search(r'^\s*I\.\s+[\u0400-\u04FF]+', text, flags=re.MULTILINE)
    if match:
        text = text[match.start():]

    # Strip end
    text = re.sub(r'\$id.*', '', text, flags=re.DOTALL)

    # Strip part headers: "Част първа", "Част втора", etc.
    text = re.sub(r'^\s*Част\s+\w+\s*$', '', text, flags=re.MULTILINE)

    # Strip numbered chapter titles: "I. Бяла черква", "II. Нещо", etc.
    text = re.sub(r'^\s*[IVXLCDM]+\.\s+.{1,100}\s*$', '', text, flags=re.MULTILINE)

    # Strip references
    text = re.sub(r'\[.*?\]', '', text, flags = re.DOTALL)

    # Strip unneeded formatting
    text = re.sub(r'^\t\* \* \*\n', '', text, flags=re.MULTILINE)
    text = re.sub(r'^\t……+\n', '', text, flags=re.MULTILINE)
    text = re.sub(r'[\t_*`«»]', '', text)

    # # Convert latin to <UNK>
    # text = re.sub(r'[A-Za-z]+', '<UNK>', text)

    # Covert digits to # (1893 -> ####)
    text = re.sub(r'\d', '#', text)

    # Convert cyrillic accented e to normal e
    text = text.replace('\u0450', '\u0435')

    # Collapse 3+ newlines to 1 (preserve paragraph breaks)
    text = re.sub(r'\n{3,}', '\n', text)

    # Strip leading/trailing whitespace
    text = text.strip()

    return text

In [100]:
def print_corpus_chars(corpus):
    print(f"Characters: {len(corpus):,}")
    print(f"Unique chars: {len(set(corpus))}")
    print(f"Sample:\n{corpus[:300]}")

In [101]:
def print_non_ascii_letters(text):
    non_ascii_letters = set()

    for ch in text:
        if unicodedata.category(ch).startswith('L') and ord(ch) > 127:
            non_ascii_letters.add((repr(ch), unicodedata.name(ch, '?')))

    for char_repr, char_name in non_ascii_letters:
        print(char_repr, char_name)

In [102]:
def open_file():
    with open(f"../data/raw/{FILE_TO_PROCESS}", "r", encoding="utf-8") as f:
        return f.read()


In [103]:
def save_file(text):
    file_to_save = FILE_TO_PROCESS.split(".")[0] + "-processed.txt"
    with open(f"../data/processed/{file_to_save}", "w", encoding="utf-8") as f:
        f.write(text)

In [104]:
raw_txt = open_file()
print_non_ascii_letters(raw_txt)

'Ч' CYRILLIC CAPITAL LETTER CHE
'ô' LATIN SMALL LETTER O WITH CIRCUMFLEX
'ö' LATIN SMALL LETTER O WITH DIAERESIS
'ѐ' CYRILLIC SMALL LETTER IE WITH GRAVE
'д' CYRILLIC SMALL LETTER DE
'ф' CYRILLIC SMALL LETTER EF
'ы' CYRILLIC SMALL LETTER YERU
'В' CYRILLIC CAPITAL LETTER VE
'ч' CYRILLIC SMALL LETTER CHE
'О' CYRILLIC CAPITAL LETTER O
'в' CYRILLIC SMALL LETTER VE
'á' LATIN SMALL LETTER A WITH ACUTE
'е' CYRILLIC SMALL LETTER IE
'н' CYRILLIC SMALL LETTER EN
'ж' CYRILLIC SMALL LETTER ZHE
'я' CYRILLIC SMALL LETTER YA
'ю' CYRILLIC SMALL LETTER YU
'ъ' CYRILLIC SMALL LETTER HARD SIGN
'Щ' CYRILLIC CAPITAL LETTER SHCHA
'м' CYRILLIC SMALL LETTER EM
'З' CYRILLIC CAPITAL LETTER ZE
'у' CYRILLIC SMALL LETTER U
'С' CYRILLIC CAPITAL LETTER ES
'щ' CYRILLIC SMALL LETTER SHCHA
'Л' CYRILLIC CAPITAL LETTER EL
'Ж' CYRILLIC CAPITAL LETTER ZHE
'л' CYRILLIC SMALL LETTER EL
'Ш' CYRILLIC CAPITAL LETTER SHA
'У' CYRILLIC CAPITAL LETTER U
'Г' CYRILLIC CAPITAL LETTER GHE
'р' CYRILLIC SMALL LETTER ER
'к' CYRILLIC SMALL L

In [105]:
cleaned_txt = clean_from_chitanka(raw_txt)
save_file(cleaned_txt)
print_corpus_chars(cleaned_txt)

Characters: 758,036
Unique chars: 136
Sample:
По гладката, стръмна южна урва на Амбарица — високия старопланински връх, който гледа над Стремска долина, ставаше нещо необикновено и чудно: върволици свят пъплеше и се катереше нагоре въз върлото. Тия върволици идеха на синджир, защото вървяха по едничката козя пътека, що извиваше на безбройни лък
